# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 human extracellular vesicle proteomics dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show main dataset title & description
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, enumerate their fields, and display their `@id`s.

In [ ]:
# List all record sets and their fields

print("Available RecordSets in this dataset:\n")
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    if hasattr(record_set, 'description'):
        print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, Name: {field.name}, Data type: {getattr(field, 'data_type', 'unknown')}")
    print()

## 3. Data Extraction
Load a record set's data into a pandas DataFrame for further analysis.

We'll use the first main record set (often the table with the biological or experimental results).

In [ ]:
# Choose the primary record set for analysis
record_sets = [record_set.id for record_set in dataset.record_sets]
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Display which record sets were loaded
print(f"Loaded RecordSets: {list(dataframes.keys())}")

# Select the first record set for demo
main_record_set_id = record_sets[0]
df = dataframes[main_record_set_id]

print(f"Columns in '{main_record_set_id}':")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Perform some basic exploratory analysis on numeric and categorical fields using Croissant field `@id`s.

- Filter rows based on a numeric field (`cr:field`/`Abundance` @id)
- Normalize a column
- Group by a categorical field (e.g., a condition or sample type)

In [ ]:
# ===
# You may want to adjust these IDs based on your RecordSet overview above!
# ===

# Find a numeric field
print("Searching for a representative numeric and category field:")
numeric_field_id = None
group_field_id = None
for field in dataset.record_sets[0].fields:
    if hasattr(field, 'data_type') and (field.data_type in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer']):
        numeric_field_id = field.id
        print(f"Numeric field candidate: @id={numeric_field_id}, name={field.name}, type={field.data_type}")
    if (not group_field_id) and (hasattr(field, 'data_type') and (field.data_type in ['Text', 'schema:Text'])):
        group_field_id = field.id
        print(f"Group-by field candidate: @id={group_field_id}, name={field.name}, type={field.data_type}")
    if numeric_field_id and group_field_id:
        break
if not numeric_field_id:
    raise Exception("No numeric field found")
if not group_field_id:
    raise Exception("No suitable group field found")

field_col = numeric_field_id

# Remove rows where the field is missing or non-numeric
df_numeric = df[pd.to_numeric(df[field_col], errors='coerce').notnull()].copy()
df_numeric[field_col] = pd.to_numeric(df_numeric[field_col], errors='coerce')

threshold = df_numeric[field_col].quantile(0.8)  # use top 20% as demo
filtered_df = df_numeric[df_numeric[field_col] > threshold]
print(f"Filtered records with {field_col} > {threshold:.3f}:")
print(filtered_df.head())

# Normalize field
filtered_df[f"{field_col}_normalized"] = (filtered_df[field_col] - filtered_df[field_col].mean()) / filtered_df[field_col].std()
print(f"\nNormalized '{field_col}' for filtered records:")
print(filtered_df[[field_col, f"{field_col}_normalized"]].head())

if group_field_id in df_numeric.columns:
    grouped_df = filtered_df.groupby(group_field_id)[field_col].mean().reset_index()
    print(f"\nGrouped data by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and the group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df_numeric[field_col].dropna(), kde=True, bins=30)
plt.title(f"Distribution of '{field_col}'")
plt.xlabel(field_col)
plt.ylabel('Frequency')
plt.show()

# Plot group field means (if grouped)
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10,4))
    order = grouped_df.sort_values(field_col, ascending=False)[group_field_id]
    sns.barplot(y=group_field_id, x=field_col, data=grouped_df, order=order)
    plt.title(f"Mean {field_col} by {group_field_id}")
    plt.xlabel(field_col)
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and analyze a Croissant-described dataset from the FAIR^2 repository using the `mlcroissant` Python package, referencing all data entities by their `@id`. You can extend this EDA and visualization for deeper biological/biomedical insights!